In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.data_processing import format_context_with_boundaries
from utils.ipa_processing import syllabify_ipa
SEP = "_"

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
df = format_context_with_boundaries(df, lang="ae")

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.28270,2.33270,2.33270,2.43270,h,ɝ,h,her,usctimit_mri_m1_146_150.avi_0.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
1,2.43270,2.64270,2.64270,2.67270,ɔ,b,ɔ_.,auburn,usctimit_mri_m1_146_150.avi_1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
2,2.64270,2.67270,2.67270,2.82270,b,ɝ,ɔ_._b,auburn,usctimit_mri_m1_146_150.avi_1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
3,2.67270,2.82270,2.82270,2.86270,ɝ,n,ɔ_._b_ɝ,auburn,usctimit_mri_m1_146_150.avi_1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
4,2.86270,2.96270,2.96270,3.19270,h,ɛ,h,hair,usctimit_mri_m1_146_150.avi_2.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
98930,22.56169,22.59172,22.59172,22.63176,m,u,m,MOVIES,usctimit_mri_f1_321_325.avi_36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
98931,22.59172,22.63176,22.63176,22.68181,u,v,m_u_.,MOVIES,usctimit_mri_f1_321_325.avi_36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
98932,22.63176,22.68181,22.68181,22.88201,v,i,m_u_._v,MOVIES,usctimit_mri_f1_321_325.avi_36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
98933,22.68181,22.88201,22.88201,23.12225,i,z,m_u_._v_i,MOVIES,usctimit_mri_f1_321_325.avi_36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1


In [4]:
with open("../../pronunciation_counts/pronunciation_counts_ae.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="ae").replace(".", "_._"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for phone1, phone2 in tqdm.tqdm(zip(df["phone1"], df["phone2"]), total=len(df)):
    bigrams.add(phone1+SEP+phone2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    phones = pronunciation.split(SEP)
    for phone in phones:
        if phone not in monogram_counts:
            monogram_counts[phone] = 0
        monogram_counts[phone] += count

100%|██████████| 48576/48576 [00:00<00:00, 965952.24it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 48576/48576 [00:00<00:00, 162526.84it/s]


In [7]:
data = {
    "bigram_surprisal": [],
    "phone2_pos_entropy": [],
}

In [8]:
data["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["phone1"] + SEP + x["phone2"]), axis=1).tolist()
data["phone2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1).tolist()
data["phone1_freq"] = df.apply(lambda x: monogram_counts.get(x["phone1"], None), axis=1).tolist()
data["phone2_freq"] = df.apply(lambda x: monogram_counts.get(x["phone2"], None), axis=1).tolist()

In [9]:
pd.DataFrame(data).to_csv("surprisal_aemri.csv", index=False)